# No-Show Prediction - Data Preprocessing & Machine Learning

## Project Objective

The objective of this notebook is to prepare the appointment dataset for machine learning and build a predictive model that identifies patients who are likely to miss their appointments.

The preprocessing pipeline includes handling missing values, feature engineering, encoding categorical variables, class imbalance treatment, model training, evaluation, and model selection.

The final model will be saved for deployment in a Streamlit application.

In [ ]:

# Import Libraries

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE

import joblib

In [ ]:
# Load Dataset

df = pd.read_csv("../data/Medical_appointment_data.csv")
df.head()

print("Dataset Loaded Successfully")

In [ ]:
print(df.shape)

print()

df.info()

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().mean()*100).round(2)
})

missing.sort_values(
    by="Percentage",
    ascending=False
)

In [ ]:
df['no_show'] = df['no_show'].map({
    'no':0,
    'yes':1
})

df['no_show'].value_counts()

In [ ]:
sns.countplot(
    data=df,
    x='no_show'
)

plt.title("Target Distribution")

plt.show()

# Feature Classification

Before preprocessing, features are categorized into numerical, categorical, binary, and target variables. This makes the preprocessing pipeline easier to manage and improves code readability.

In [ ]:

# Feature Classification


target = 'no_show'

numerical_features = [
    'age',
    'appointment_time',
    'average_temp_day',
    'max_temp_day',
    'average_rain_day',
    'max_rain_day'
]

categorical_features = [
    'gender',
    'specialty',
    'disability',
    'place',
    'appointment_shift',
    'rain_intensity',
    'heat_intensity'
]

binary_features = [
    'under_12_years_old',
    'over_60_years_old',
    'patient_needs_companion',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'Scholarship',
    'SMS_received',
    'rainy_day_before',
    'storm_day_before'
]

date_feature = 'appointment_date_continuous'

print("Numerical Features :", len(numerical_features))
print("Categorical Features :", len(categorical_features))
print("Binary Features :", len(binary_features))

In [ ]:
print("Numerical Features")
print(df[numerical_features].isnull().sum())

print("\nCategorical Features")
print(df[categorical_features].isnull().sum())

print("\nBinary Features")
print(df[binary_features].isnull().sum())

# Missing Value Treatment Strategy

Based on the exploratory data analysis, missing values will be handled as follows:

| Feature Type | Strategy |
|--------------|----------|
| Numerical Features | Median Imputation |
| Categorical Features | Replace with "Unknown" |
| Binary Features | No Missing Values |
| Target Variable | No Missing Values |

Median is selected instead of mean because it is more robust to outliers. Categorical missing values are replaced with "Unknown" to preserve information rather than deleting records.

In [ ]:
# ===============================
# Numerical Features
# ===============================

for col in numerical_features:
    df[col].fillna(df[col].median(), inplace=True)

# ===============================
# Categorical Features
# ===============================

for col in categorical_features:
    df[col].fillna("Unknown", inplace=True)

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Convert to datetime
df['appointment_date_continuous'] = pd.to_datetime(
    df['appointment_date_continuous']
)

# Create temporal features
df['Year'] = df['appointment_date_continuous'].dt.year
df['Month'] = df['appointment_date_continuous'].dt.month
df['Day'] = df['appointment_date_continuous'].dt.day
df['DayOfWeek'] = df['appointment_date_continuous'].dt.dayofweek
df['Quarter'] = df['appointment_date_continuous'].dt.quarter

In [ ]:
df.drop(
    columns=['appointment_date_continuous'],
    inplace=True
)

In [ ]:
df.info()

# Feature and Target Separation

The dataset is divided into predictor variables (X) and the target variable (y). This separation is required before splitting the dataset into training and testing sets.

In [ ]:
# ===============================
# Feature and Target
# ===============================

X = df.drop('no_show', axis=1)
y = df['no_show']

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

In [ ]:
categorical_columns = [
    'specialty',
    'gender',
    'disability',
    'place',
    'appointment_shift',
    'rain_intensity',
    'heat_intensity'
]

numerical_columns = [
    'appointment_time',
    'age',
    'average_temp_day',
    'average_rain_day',
    'max_temp_day',
    'max_rain_day'
]

binary_columns = [
    'under_12_years_old',
    'over_60_years_old',
    'patient_needs_companion',
    'rainy_day_before',
    'storm_day_before',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'Scholarship',
    'SMS_received',
    'Year',
    'Month',
    'Day',
    'DayOfWeek',
    'Quarter'
]
print("Categorical Features")
print(categorical_columns)

print("\nNumerical Features")
print(numerical_columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    shuffle=False
)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

In [ ]:
print("Training")

print(y_train.value_counts(normalize=True) * 100)

print()

print("Testing")

print(y_test.value_counts(normalize=True) * 100)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_columns),
    ('cat', categorical_transformer, categorical_columns),
    ('bin', binary_transformer, binary_columns)
])
print("Preprocessing pipeline created successfully.")

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_processed,
    y_train
)

print("Before SMOTE")

print(y_train.value_counts())

print()

print("After SMOTE")

print(y_train_balanced.value_counts())

# Model Training and Evaluation

Four machine learning algorithms are trained and compared for predicting patient no-shows.

The models evaluated are:

1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM

Each model is evaluated using:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC Score
- Confusion Matrix
- Classification Report

The best-performing model will be selected based on F1-Score and ROC-AUC, as these metrics are most appropriate for imbalanced healthcare datasets.

In [ ]:
results = []

# Model 1 – Logistic Regression

Logistic Regression is selected as the baseline model because it is simple, interpretable, and computationally efficient.

Although it assumes a linear relationship between the predictors and the target, it provides a useful benchmark for comparing more complex ensemble models.

In [ ]:
log_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_model.fit(
    X_train_balanced,
    y_train_balanced
)

In [ ]:
y_pred_log = log_model.predict(X_test_processed)

y_prob_log = log_model.predict_proba(X_test_processed)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred_log)
precision = precision_score(y_test, y_pred_log)
recall = recall_score(y_test, y_pred_log)
f1 = f1_score(y_test, y_pred_log)
roc = roc_auc_score(y_test, y_prob_log)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC : {roc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred_log)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Logistic Regression Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(y_test, y_pred_log))

In [ ]:
results.append({
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1,
    "ROC AUC":roc
})

## Logistic Regression Performance Summary

The Logistic Regression model serves as the baseline classifier for predicting appointment no-shows.

### Performance Summary

- Accuracy: **66.61%**
- Precision: **47.91%**
- Recall: **83.00%**
- F1-Score: **60.75%**
- ROC-AUC: **75.96%**

### Interpretation

The model demonstrates excellent recall, correctly identifying most patients who are likely to miss their appointments. This characteristic is particularly valuable in healthcare because missing high-risk patients can result in lost revenue and inefficient resource utilization.

However, the relatively low precision indicates that many patients are incorrectly classified as high-risk, leading to unnecessary reminder interventions.

Since Logistic Regression assumes a linear relationship between features and the target variable, it may not capture the complex behavioral patterns present in patient attendance data. Therefore, more advanced ensemble models are expected to achieve better predictive performance.

## Model Interpretation

Logistic Regression establishes a baseline performance for the no-show prediction problem.

The model provides high interpretability and enables healthcare administrators to understand the overall relationship between patient characteristics and appointment attendance.

However, because patient behavior is influenced by complex and non-linear interactions, more advanced ensemble models are expected to achieve higher predictive performance.

# Model 2 – Random Forest

Random Forest is an ensemble machine learning algorithm that combines multiple decision trees to improve predictive performance.

Unlike Logistic Regression, Random Forest can capture complex non-linear relationships between patient demographics, appointment characteristics, weather conditions, and attendance behavior.

This model is expected to outperform the baseline Logistic Regression model.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_balanced, y_train_balanced)

## Random Forest Performance Summary

The Random Forest model achieved higher overall accuracy and precision compared to Logistic Regression. However, it showed a noticeable reduction in recall and F1-score.

### Performance

- Accuracy: **71.49%**
- Precision: **54.76%**
- Recall: **48.48%**
- F1-Score: **51.43%**
- ROC-AUC: **75.58%**

### Interpretation

Although Random Forest correctly classified more overall appointments, it failed to identify a large proportion of actual no-show patients. Since the primary business objective is to identify patients at high risk of missing appointments for targeted interventions, recall and F1-score are more critical than overall accuracy.

Therefore, despite its higher accuracy, Random Forest does not outperform the baseline Logistic Regression model for this healthcare application.

In [ ]:
rf_model.fit(X_train_balanced, y_train_balanced)

y_pred_rf = rf_model.predict(X_test_processed)

y_prob_rf = rf_model.predict_proba(X_test_processed)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf)
recall = recall_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)
roc = roc_auc_score(y_test, y_prob_rf)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC : {roc:.4f}")

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_rf
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Random Forest Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred_rf
))

In [ ]:
results.append({

    "Model":"Random Forest",

    "Accuracy":accuracy,

    "Precision":precision,

    "Recall":recall,

    "F1 Score":f1,

    "ROC AUC":roc

})

## Random Forest Performance Summary

Random Forest captures complex interactions between patient demographics, appointment details, weather conditions, and health-related factors.

Compared with Logistic Regression, Random Forest is expected to improve prediction quality by modeling non-linear decision boundaries.

If the model achieves a higher F1-score and ROC-AUC, it demonstrates that patient attendance behavior is influenced by complex feature interactions rather than simple linear relationships.

This makes Random Forest a strong candidate for deployment in the rehabilitation center's appointment management system.

# Model 3 – XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is an advanced ensemble learning algorithm that builds decision trees sequentially to correct the mistakes of previous trees.

Compared to Random Forest, XGBoost generally provides better predictive performance, handles missing values efficiently, and captures complex non-linear relationships between variables.

Due to its excellent performance on structured datasets, XGBoost is widely used in healthcare analytics, fraud detection, and predictive modeling.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(
    X_train_balanced,
    y_train_balanced
)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_processed)

y_prob_xgb = xgb_model.predict_proba(
    X_test_processed
)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred_xgb)

precision = precision_score(y_test, y_pred_xgb)

recall = recall_score(y_test, y_pred_xgb)

f1 = f1_score(y_test, y_pred_xgb)

roc = roc_auc_score(y_test, y_prob_xgb)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC : {roc:.4f}")

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_xgb
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("XGBoost Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred_xgb
))

In [ ]:
results.append({

    "Model":"XGBoost",

    "Accuracy":accuracy,

    "Precision":precision,

    "Recall":recall,

    "F1 Score":f1,

    "ROC AUC":roc

})

## XGBoost Performance Summary

The XGBoost model uses gradient boosting to iteratively improve prediction accuracy by correcting errors made by previous decision trees.

Its ability to capture complex interactions among demographic, clinical, environmental, and appointment-related features makes it well suited for predicting patient no-shows.

The model performance will be compared against Logistic Regression and Random Forest to determine the most suitable algorithm for deployment.

## XGBoost Performance Summary

The XGBoost classifier achieved the highest overall accuracy and ROC-AUC among the models evaluated so far, indicating strong discriminative ability.

### Performance

- Accuracy: **73.04%**
- Precision: **57.50%**
- Recall: **51.37%**
- F1-Score: **54.26%**
- ROC-AUC: **78.18%**

### Interpretation

The model effectively distinguishes between patients who attend and those who miss appointments, as reflected by its high ROC-AUC score. However, the relatively lower recall indicates that a substantial number of actual no-show patients remain unidentified.

While XGBoost provides the highest overall predictive accuracy, Logistic Regression continues to achieve superior recall, making it more suitable when the primary objective is to identify as many high-risk patients as possible for targeted interventions.

# Model 4 – LightGBM Classifier

LightGBM (Light Gradient Boosting Machine) is a high-performance gradient boosting framework designed for speed and efficiency.

Compared with traditional boosting algorithms, LightGBM grows trees leaf-wise rather than level-wise, allowing it to capture complex patterns while reducing training time.

LightGBM is widely used in healthcare analytics because it performs exceptionally well on structured datasets with both numerical and categorical features.

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

lgbm_model.fit(
    X_train_balanced,
    y_train_balanced
)

In [ ]:
y_pred_lgbm = lgbm_model.predict(X_test_processed)

y_prob_lgbm = lgbm_model.predict_proba(
    X_test_processed
)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred_lgbm)

precision = precision_score(y_test, y_pred_lgbm)

recall = recall_score(y_test, y_pred_lgbm)

f1 = f1_score(y_test, y_pred_lgbm)

roc = roc_auc_score(y_test, y_prob_lgbm)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC : {roc:.4f}")

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_lgbm
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Purples"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("LightGBM Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred_lgbm
))

In [ ]:
results.append({

    "Model":"LightGBM",

    "Accuracy":accuracy,

    "Precision":precision,

    "Recall":recall,

    "F1 Score":f1,

    "ROC AUC":roc

})

## LightGBM Performance Summary

LightGBM is designed to efficiently model complex relationships between patient demographics, health conditions, appointment characteristics, and environmental factors.

Its leaf-wise tree growth strategy allows it to achieve high predictive performance while maintaining computational efficiency.

The model will be evaluated against Logistic Regression, Random Forest, and XGBoost using Accuracy, Precision, Recall, F1-score, and ROC-AUC to determine the most suitable classifier for deployment.

## LightGBM Performance Summary

The LightGBM classifier demonstrated the best balance between precision and recall among the tree-based models evaluated.

### Performance

- Accuracy: **73.03%**
- Precision: **56.64%**
- Recall: **57.04%**
- F1-Score: **56.84%**
- ROC-AUC: **78.37%**

### Interpretation

LightGBM achieved the highest ROC-AUC score among all evaluated models, demonstrating excellent ability to distinguish between patients who attend appointments and those who miss them.

Compared with Random Forest and XGBoost, LightGBM provides a better balance between identifying high-risk patients and minimizing false alarms. Although the F1-score remains below the desired project target, LightGBM shows the strongest overall performance among the ensemble learning methods.

# Model Comparison

Four machine learning algorithms were trained and evaluated for predicting patient no-shows.

The models were compared using:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

The best model will be selected based on overall business requirements and predictive performance.

In [ ]:
results_df = pd.DataFrame(results)

results_df

In [ ]:
results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

In [ ]:
metrics = ["Accuracy","Precision","Recall","F1 Score","ROC AUC"]

results_df.set_index("Model")[metrics].plot(
    kind="bar",
    figsize=(12,6)
)

plt.title("Comparison of Classification Models")

plt.ylabel("Score")

plt.xticks(rotation=0)

plt.grid(axis='y')

plt.show()

In [ ]:
best_model = results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

best_model

## Best Performing Model

Among all evaluated classifiers, LightGBM achieved the highest ROC-AUC score while maintaining a balanced trade-off between precision and recall.

Although Logistic Regression achieved the highest recall, LightGBM demonstrated superior overall discriminative ability, making it the preferred model for deployment.

# Feature Importance Analysis

Feature importance helps identify which variables contribute the most to predicting patient no-shows.

Understanding these factors enables the rehabilitation center to:

- Identify high-risk patients
- Improve appointment scheduling
- Optimize staffing
- Design targeted reminder strategies
- Reduce financial loss caused by missed appointments

In [ ]:
# Numerical features
num_features = numerical_features

# Categorical features after OneHotEncoding
cat_features = preprocessor.named_transformers_["cat"]\
                           .named_steps["encoder"]\
                           .get_feature_names_out(categorical_features)

# Binary features
bin_features = binary_features

# Combine all feature names
feature_names = list(num_features) + list(cat_features) + list(bin_features)

print("Total Features :", len(feature_names))
print("Model Features :", len(lgbm_model.feature_importances_))

In [ ]:
print(X_train.columns.tolist())

In [ ]:
print(len(X_train.columns))

In [ ]:
print(categorical_features)
print(len(categorical_features))

In [ ]:
print(numerical_features)
print(len(numerical_features))

In [ ]:
print(binary_features)
print(len(binary_features))

# Feature Importance

Feature importance identifies which variables contribute the most to predicting patient no-shows.

The LightGBM model automatically calculates feature importance based on how frequently and effectively each feature is used when constructing decision trees.

The most influential features provide valuable business insights for reducing appointment no-shows.

In [ ]:
from lightgbm import plot_importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

plot_importance(
    lgbm_model,
    max_num_features=20,
    importance_type="gain",
    figsize=(10,8)
)

plt.title("Top 20 Important Features - LightGBM")

plt.tight_layout()

plt.show()

In [ ]:
plot_importance(lgbm_model)

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": lgbm_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

In [ ]:
print("Feature Names :", len(feature_names))
print("Model Importances :", len(lgbm_model.feature_importances_))

In [ ]:
print(lgbm_model.feature_importances_.shape)

In [ ]:
print(X_train_balanced.shape)

In [ ]:
print("Feature Names :", len(feature_names))
print("Model Importances :", len(lgbm_model.feature_importances_))
print(X_train_balanced.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(len(lgbm_model.feature_importances_))

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": lgbm_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

In [ ]:
print(type(feature_names))
print(feature_names.shape)
print(len(feature_names))

print(type(lgbm_model.feature_importances_))
print(lgbm_model.feature_importances_.shape)
print(len(lgbm_model.feature_importances_))

## Feature Importance Analysis

The LightGBM model identified several key factors influencing appointment attendance.

### Major Observations

- Weather-related variables such as Average Temperature, Maximum Temperature, Average Rainfall, and Maximum Rainfall were among the strongest predictors.
- Patient Age contributed significantly to no-show prediction.
- Appointment Time and calendar-related features (Day, Month, Day of Week) also played an important role.
- Geographic location (Place) influenced attendance behaviour, indicating regional differences.
- Specialty and Gender showed moderate predictive importance.

### Business Interpretation

The results suggest that both patient characteristics and external environmental conditions influence appointment attendance. Hospitals can leverage these insights to proactively identify high-risk appointments and optimize scheduling, reminder systems, and staffing decisions.